<img src="https://raw.githubusercontent.com/ProteinDesignLab/caliby/main/assets/caliby_transparent.jpg" alt="Caliby" width="175"/>

# Caliby: Ensemble-conditioned protein sequence design

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProteinDesignLab/caliby/blob/main/notebooks/Caliby.ipynb)

Caliby is a Potts model-based protein sequence design method that can condition on structural ensembles. For more details, read our preprint: [Ensemble-conditioned protein sequence design with Caliby](https://www.biorxiv.org/content/10.1101/2025.09.30.679633v4) (Shuai et al., 2025).

This notebook demonstrates the core functionality:
1. **Fixed-backbone sequence design** - Design new sequences for a protein backbone, with optional positional constraints
2. **Self-consistency evaluation** - Predict structures for designed sequences with single-sequence AlphaFold2 and compare to the designed backbone
3. **Ensemble generation** - Sample synthetic backbones with Protpardelle-1c partial diffusion
4. **Ensemble-conditioned design** - Design sequences against a structural ensemble

**Requirements:** A GPU runtime is required. Go to *Runtime > Change runtime type > T4 GPU*.

In [ ]:
#@title Install Caliby
#@markdown This will install Caliby and its dependencies (including AlphaFold2 for self-consistency evaluation).

import os, subprocess, sys

if not os.path.isfile("CALIBY_READY"):
    print("Installing Caliby...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "caliby[af2] @ git+https://github.com/ProteinDesignLab/caliby.git",
         "molview>=0.1.0",
         "--quiet"],
        check=True,
    )
    # Mark installation complete so re-runs skip this step.
    with open("CALIBY_READY", "w") as f:
        f.write("done")


print("Caliby is installed.")

# Verify GPU is available.
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")

# Input structure

**Option A:** Run the next cell to upload your own PDB/CIF file.

**Option B:** Skip the next cell and go straight to **Prepare input structure** — the example structure (PDB 8sot) will be used automatically.

In [ ]:
#@title Upload your own PDB (optional)
#@markdown Run this cell to upload a PDB or CIF file. **Skip this cell to use the example structure.**
import os, shutil

INPUT_DIR = "input_pdbs"
if os.path.exists(INPUT_DIR):
    shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR)

from google.colab import files
print("Upload a PDB or CIF file:")
uploaded = files.upload()
for fn in uploaded:
    os.rename(fn, os.path.join(INPUT_DIR, fn))
print(f"Uploaded {len(uploaded)} file(s).")

In [ ]:
#@title Prepare input structure
#@markdown Downloads the example PDB if no file was uploaded. Cleans and standardizes the structure.
import os, glob, shutil, urllib.request, base64

INPUT_DIR = "input_pdbs"
os.makedirs(INPUT_DIR, exist_ok=True)

pdb_files = glob.glob(os.path.join(INPUT_DIR, "*.pdb")) + glob.glob(os.path.join(INPUT_DIR, "*.cif"))

if not pdb_files:
    BASE_URL = "https://raw.githubusercontent.com/ProteinDesignLab/caliby/main/examples/example_data/native_pdbs"
    print("No uploaded file found — downloading example structure (8sot.cif)...")
    urllib.request.urlretrieve(f"{BASE_URL}/8sot.cif", os.path.join(INPUT_DIR, "8sot.cif"))
    pdb_files = glob.glob(os.path.join(INPUT_DIR, "*.cif"))

pdb_file = pdb_files[0]

# Clean the PDB to standardize it for downstream use.
from caliby import clean_pdbs
from IPython.display import display, HTML

CLEAN_DIR = "cleaned_pdbs"
if os.path.exists(CLEAN_DIR):
    shutil.rmtree(CLEAN_DIR)

cleaned_paths = clean_pdbs([pdb_file], out_dir=CLEAN_DIR)
pdb_file = cleaned_paths[0]
pdb_name = os.path.splitext(os.path.basename(pdb_file))[0]

cleaned_b64 = base64.b64encode(open(pdb_file, "rb").read()).decode()
cleaned_filename = os.path.basename(pdb_file)
display(HTML(
    "<div style='margin: 10px 0; padding: 12px 16px; background: #fff8e1; border-left: 4px solid #ffb300; border-radius: 4px;'>"
    "<b>Note:</b> Your structure has been automatically cleaned and standardized to mmCIF format. "
    "This removes unsupported residues, unresolved atoms, and fixes blank chain IDs to avoid "
    "downstream parsing issues. <b>If you plan to use positional constraints, please download "
    "the cleaned file and verify the residue numbering.</b><br><br>"
    f"<a href='data:chemical/x-cif;base64,{cleaned_b64}' download='{cleaned_filename}' "
    "style='display: inline-block; padding: 6px 14px; background: #ffb300; color: #fff; "
    "border-radius: 4px; text-decoration: none; font-weight: 600;'>Download cleaned file</a>"
    "</div>"
))

In [ ]:
#@title Visualize input structure
import os

import molview as mv
from IPython.display import display, HTML
from atomworks.io.utils import non_rcsb

from caliby.eval.eval_utils.seq_des_utils import preprocess_pdb


def _format_sequence_blocks(chain_info, line_width=80):
    blocks = []
    for chain_name, info in chain_info.items():
        sequence = info["processed_entity_canonical_sequence"]
        if not sequence:
            continue
        wrapped = "<br>".join(sequence[i : i + line_width] for i in range(0, len(sequence), line_width))
        blocks.append(
            f"<div style='margin: 0 0 12px 0;'>"
            f"<div style='font-weight: 600; margin-bottom: 4px;'>Chain {chain_name} ({len(sequence)} aa)</div>"
            f"<pre style='margin: 0; padding: 10px 12px; background: #f7f7f7; border: 1px solid #ddd; border-radius: 6px; white-space: pre-wrap; word-break: break-word; font-family: monospace; line-height: 1.4;'>{wrapped}</pre>"
            f"</div>"
        )
    return "".join(blocks) or "<p>No protein sequence could be extracted from the parsed structure.</p>"


pdb_data = open(pdb_file).read()
v = mv.view(width=840, height=500)
v.addModel(pdb_data, name=pdb_name)
v.setColorMode("rainbow")
v.setBackgroundColor("#F2F2F2")
display(HTML(f"<h4>Input structure: {os.path.basename(pdb_file)}</h4>"))
display(HTML(v._repr_html_()))

example = preprocess_pdb(pdb_file, data_cfg=None)
chain_info = non_rcsb.initialize_chain_info_from_atom_array(example["atom_array"])
display(HTML("<h4>Input sequence</h4>"))
display(HTML(_format_sequence_blocks(chain_info)))

In [ ]:
#@title Load model
#@markdown Loads the Caliby model. Weights are downloaded automatically on first use.
#@markdown
#@markdown | Model | Description |
#@markdown |-------|-------------|
#@markdown | **Caliby** | Default model trained on all PDB chains with 0.3Å Gaussian noise (monomers only) |
#@markdown | **SolubleCaliby** | Excludes transmembrane proteins (similar to SolubleMPNN, [Goverde et al., 2024](https://www.nature.com/articles/s41586-024-07601-y)), monomers only |
#@markdown | **SolubleCaliby v1** | SolubleCaliby trained on both monomers and interfaces |

model_name = "soluble_caliby_v1"  #@param ["caliby", "soluble_caliby", "soluble_caliby_v1"]

import logging, os, warnings
logging.getLogger("transforms").setLevel(logging.INFO)

_SUBPROCESS_WARNING_FILTERS = [
    "ignore::DeprecationWarning:jupyter_client.session",
    "ignore::DeprecationWarning:jax._src.linear_util",
    "ignore::DeprecationWarning:colabdesign.shared.model",
    "ignore::DeprecationWarning:colabdesign.shared.utils",
]

def configure_notebook_warning_filters():
    warnings.filterwarnings("ignore", message=".*secret.*HF_TOKEN.*")
    warnings.filterwarnings("ignore", message=".*torch.jit.script_method.*is deprecated.*")
    warnings.filterwarnings(
        "ignore",
        category=DeprecationWarning,
        message=r".*datetime\.datetime\.utcnow\(\) is deprecated.*",
    )
    warnings.filterwarnings(
        "ignore",
        category=DeprecationWarning,
        message=r".*use of fork\(\) may lead to deadlocks in the child\..*",
    )
    warnings.filterwarnings(
        "ignore",
        category=RuntimeWarning,
        message=r".*os\.fork\(\) was called\..*multithreaded code.*JAX is multithreaded.*",
    )
    warnings.filterwarnings(
        "ignore",
        category=DeprecationWarning,
        message=r".*jax\.lib\.xla_bridge\.get_backend is deprecated.*",
    )
    warnings.filterwarnings(
        "ignore",
        category=DeprecationWarning,
        message=r".*optax\.dpsgd is deprecated: use optax\.contrib\.dpsgd.*",
    )
    warnings.filterwarnings(
        "ignore",
        category=DeprecationWarning,
        message=r".*Passing arguments 'a', 'a_min' or 'a_max' to jax\.numpy\.clip is deprecated.*",
    )
    for module in (
        r"jupyter_client\.session",
        r"jax\._src\.linear_util",
        r"colabdesign\.shared\.model",
        r"colabdesign\.shared\.utils",
        r"multiprocessing\.popen_fork",
    ):
        warnings.filterwarnings("ignore", category=DeprecationWarning, module=module)

    existing = [item for item in os.environ.get("PYTHONWARNINGS", "").split(",") if item]
    for item in reversed(_SUBPROCESS_WARNING_FILTERS):
        if item not in existing:
            existing.insert(0, item)
    os.environ["PYTHONWARNINGS"] = ",".join(existing)

# Some ML imports reset warning filters, so rerun this helper after loading the model.
configure_notebook_warning_filters()

from caliby import load_model

def _get_available_num_workers():
    if hasattr(os, "sched_getaffinity"):
        return max(1, len(os.sched_getaffinity(0)))
    return max(1, os.cpu_count() or 1)

num_workers = _get_available_num_workers()
model = load_model(model_name)
configure_notebook_warning_filters()
print(f"Model '{model_name}' loaded on {model.device}.")
print(f"Using {num_workers} data-loading workers.")

# Fixed-backbone sequence design

In [ ]:
#@title Run sequence design
#@markdown ### Sampling parameters
num_seqs = 4  #@param {type:"integer"}
temperature = 0.01  #@param {type:"number"}
omit_aas = ""  #@param {"type":"string","placeholder":"e.g. C,M"}
#@markdown ### Constraints (optional)
#@markdown Enable and fill in the fields you need. Positions use `label_seq_id` numbering.
#@markdown <details><summary><b>Constraint reference</b></summary>
#@markdown
#@markdown | Constraint | Format | Description |
#@markdown |---|---|---|
#@markdown | `fixed_pos_seq` | `A1-10,A20-30` | Fix sequence positions |
#@markdown | `fixed_pos_scn` | `A1-10` | Also condition on sidechains (subset of fixed_pos_seq) |
#@markdown | `fixed_pos_override_seq` | `A26:A,A27:L` | Override then fix position to specific AA |
#@markdown | `pos_restrict_aatype` | `A26:AVG,A27:VG` | Restrict allowed AAs at positions |
#@markdown | `symmetry_pos` | `A10,B10\|A11,B11` | Tie sampling across positions |
#@markdown
#@markdown </details>
fixed_pos_seq = ""  #@param {"type":"string","placeholder":"e.g. A1-100,B1-100"}
fixed_pos_scn = ""  #@param {"type":"string","placeholder":"e.g. A1-10,A12,A15-20"}
fixed_pos_override_seq = ""  #@param {"type":"string","placeholder":"e.g. A26:A,A27:L"}
pos_restrict_aatype = ""  #@param {"type":"string","placeholder":"e.g. A26:AVG,A27:VG"}
symmetry_pos = ""  #@param {"type":"string","placeholder":"e.g. A10,B10,C10|A11,B11,C11"}

import pandas as pd
from IPython.display import display, HTML
from caliby import make_constraints

omit_list = [aa.strip() for aa in omit_aas.split(",") if aa.strip()] or None

sample_kwargs = dict(
    pdb_paths=[pdb_file],
    num_seqs_per_pdb=num_seqs,
    batch_size=1,
    temperature=temperature,
    omit_aas=omit_list,
    num_workers=num_workers,
    out_dir="outputs/seq_des",
)

constraint_inputs = {
    "fixed_pos_seq": fixed_pos_seq,
    "fixed_pos_scn": fixed_pos_scn,
    "fixed_pos_override_seq": fixed_pos_override_seq,
    "pos_restrict_aatype": pos_restrict_aatype,
    "symmetry_pos": symmetry_pos,
}
constraints = {k: v for k, v in constraint_inputs.items() if v.strip()}
if constraints:
    sample_kwargs["pos_constraint_df"] = make_constraints({pdb_name: constraints})

configure_notebook_warning_filters()
results = model.sample(**sample_kwargs)

df = pd.DataFrame(results)
title = f"Designed {len(df)} sequences"
display(HTML(f"<h3>{title}</h3>"))
print(f"Output CIF files saved to outputs/seq_des/")
print(f"Available columns: {list(df.columns)}\n")

df_display = df[["example_id", "seq", "U"]].copy()
df_display["U"] = df_display["U"].round(2)
df_display["seq"] = df_display["seq"].str[:40] + df_display["seq"].apply(
    lambda s: f"... ({len(s)} aa)" if len(s) > 40 else ""
)
display(df_display.style.set_properties(**{
    "text-align": "left",
    "font-family": "monospace",
}).set_properties(subset=["U"], **{
    "font-weight": "bold",
}).hide(axis="index"))

In [ ]:
#@title Download sequence design outputs
#@markdown Downloads the single-backbone design outputs from `outputs/seq_des/`.

from pathlib import Path
import zipfile
from google.colab import files

outputs_root = Path("outputs")
output_dirs = [outputs_root / "seq_des"]
missing_dirs = [str(path) for path in output_dirs if not path.exists()]
if missing_dirs:
    raise FileNotFoundError(f"Run sequence design first to create: {', '.join(missing_dirs)}")

# Save results DataFrame as CSV
df.to_csv(outputs_root / "seq_des" / "seq_des_outputs.csv", index=False)

archive_path = Path("caliby_seq_des_results.zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for output_dir in output_dirs:
        for path in output_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(outputs_root))

files.download(str(archive_path))
print("Download started. Check your browser's downloads.")

## Self-consistency evaluation

Refold each designed sequence with AlphaFold2 and compare the predicted structure to the designed backbone. Lower `sc_ca_rmsd` and higher `tmalign_score` indicate the designed sequence folds back to the intended structure. The first run downloads AF2 weights.

In [ ]:
#@title Run self-consistency evaluation
#@markdown Refolds designed sequences with AlphaFold2 and compares to the designed backbone.
#@markdown The first run may download AF2 weights.
num_recycles = 3  #@param {type:"integer"}

from pathlib import Path

import pandas as pd
from IPython.display import display, HTML

designed_pdbs = df["out_pdb"].tolist()

configure_notebook_warning_filters()
sc_results = model.self_consistency_eval(
    designed_pdbs,
    out_dir="outputs/seq_des_sc",
    num_models=5,
    num_recycles=num_recycles,
)

sc_df = pd.DataFrame([
    {"sample_id": k, **v} for k, v in sc_results.items()
])
df_sc = df.copy()
df_sc["sample_id"] = df_sc["out_pdb"].apply(lambda path: Path(path).stem)
df_sc = df_sc.merge(sc_df, on="sample_id", how="left", validate="one_to_one")

display(HTML("<h3>Self-consistency evaluation</h3>"))
print(f"AF2 predictions saved to outputs/seq_des_sc/\n")

df_display = df_sc[["sample_id", "example_id", "U", "sc_ca_rmsd", "avg_ca_plddt", "tmalign_score"]].copy()
df_display[["U", "sc_ca_rmsd", "tmalign_score"]] = df_display[["U", "sc_ca_rmsd", "tmalign_score"]].round(3)
df_display["avg_ca_plddt"] = df_display["avg_ca_plddt"].round(1)
display(df_display.style.set_properties(**{
    "text-align": "left",
    "font-family": "monospace",
}).hide(axis="index"))

# Show 3D viewer for best AF2 prediction (by TM-align score).
import molview as mv

valid_scores = df_sc["tmalign_score"].dropna()
if not valid_scores.empty:
    best_idx = valid_scores.idxmax()
    best_sample = df_sc.loc[best_idx, "sample_id"]
    best_af2_pdb = Path(f"outputs/seq_des_sc/struct_preds/af2_{best_sample}.pdb")

    if best_af2_pdb.exists():
        pdb_data = best_af2_pdb.read_text()
        v = mv.view(width=840, height=500)
        v.addModel(pdb_data, name=f"AF2: {best_sample}")
        v.setColorMode("plddt")
        v.setBackgroundColor("#F2F2F2")
        display(HTML(f"<h4>AF2 prediction: {best_sample} (pLDDT coloring)</h4>"))
        display(HTML(v._repr_html_()))
else:
    print("TM-align scores unavailable; skipping AF2 viewer.")

In [ ]:
#@title Download self-consistency outputs
#@markdown Downloads only the AlphaFold2 self-consistency outputs.

from pathlib import Path
import zipfile
from google.colab import files

outputs_root = Path("outputs")
sc_dir = outputs_root / "seq_des_sc"
aligned_dir = sc_dir / "ca_aligned_struct_preds"
metrics_path = sc_dir / "self_consistency_metrics.csv"
required_paths = [sc_dir, aligned_dir]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Run self-consistency first to create: {', '.join(missing_paths)}")

# Save self-consistency metrics as CSV
df_sc.to_csv(metrics_path, index=False)

archive_path = Path("self_consistency_results.zip")
archive_root = Path("self_consistency_results")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(metrics_path, archive_root / metrics_path.name)
    for path in aligned_dir.rglob("*"):
        if path.is_file():
            zf.write(path, archive_root / path.relative_to(sc_dir))

files.download(str(archive_path))
print("Download started. Check your browser's downloads.")

# Ensemble generation

Generate a synthetic backbone ensemble with Protpardelle-1c partial diffusion. The first run may download Protpardelle-1c weights. For best ensemble-conditioned design results, aim for 16-32 conformers when runtime allows.

In [ ]:
#@title Generate backbone ensemble
#@markdown Generates a synthetic ensemble with Protpardelle-1c partial diffusion.
#@markdown The first run may download Protpardelle-1c weights. For best results, aim for 16-32 conformers when runtime allows.
ensemble_num_samples = 16  #@param {type:"integer"}
ensemble_batch_size = 4  #@param {type:"integer"}

from pathlib import Path

import pandas as pd
from IPython.display import display, HTML
from caliby import generate_ensembles

ensemble_generation_out_dir = "outputs/generate_ensembles"
generated_pdb_to_conformers = generate_ensembles(
    pdb_paths=[pdb_file],
    out_dir=ensemble_generation_out_dir,
    num_samples_per_pdb=ensemble_num_samples - 1,
    batch_size=ensemble_batch_size,
)
generated_conformers = generated_pdb_to_conformers.get(pdb_name, [])
if not generated_conformers:
    raise RuntimeError(f"No conformers were generated for {pdb_name}.")

all_conformers = [pdb_file] + generated_conformers

print(f"Generated {len(generated_conformers)} new conformers for ensemble-conditioned design.")


# Ensemble-conditioned design

Design against the original input structure plus generated conformers. Constraints use the primary conformer, which is always the uploaded or example structure. If you hit a residue index or chain ID mismatch error, clean the input structure before ensemble generation so all conformers stay aligned.

In [ ]:
#@title Run ensemble-conditioned design
#@markdown ### Sampling parameters
num_seqs = 4  #@param {type:"integer"}
temperature = 0.01  #@param {type:"number"}
omit_aas = ""  #@param {"type":"string","placeholder":"e.g. C,M"}
#@markdown ### Constraints (optional)
#@markdown Enable and fill in the fields you need. Positions use `label_seq_id` numbering relative to the primary conformer.
#@markdown <details><summary><b>Constraint reference</b></summary>
#@markdown
#@markdown | Constraint | Format | Description |
#@markdown |---|---|---|
#@markdown | `fixed_pos_seq` | `A1-10,A20-30` | Fix sequence positions |
#@markdown | `fixed_pos_scn` | `A1-10` | Also condition on sidechains (subset of fixed_pos_seq) |
#@markdown | `fixed_pos_override_seq` | `A26:A,A27:L` | Override then fix position to specific AA |
#@markdown | `pos_restrict_aatype` | `A26:AVG,A27:VG` | Restrict allowed AAs at positions |
#@markdown | `symmetry_pos` | `A10,B10\|A11,B11` | Tie sampling across positions |
#@markdown
#@markdown </details>
fixed_pos_seq = ""  #@param {"type":"string","placeholder":"e.g. A1-100,B1-100"}
fixed_pos_scn = ""  #@param {"type":"string","placeholder":"e.g. A1-10,A12,A15-20"}
fixed_pos_override_seq = ""  #@param {"type":"string","placeholder":"e.g. A26:A,A27:L"}
pos_restrict_aatype = ""  #@param {"type":"string","placeholder":"e.g. A26:AVG,A27:VG"}
symmetry_pos = ""  #@param {"type":"string","placeholder":"e.g. A10,B10,C10|A11,B11,C11"}

from pathlib import Path

import pandas as pd
from IPython.display import display, HTML
from caliby import make_ensemble_constraints

omit_list = [aa.strip() for aa in omit_aas.split(",") if aa.strip()] or None

constraint_inputs = {
    "fixed_pos_seq": fixed_pos_seq,
    "fixed_pos_scn": fixed_pos_scn,
    "fixed_pos_override_seq": fixed_pos_override_seq,
    "pos_restrict_aatype": pos_restrict_aatype,
    "symmetry_pos": symmetry_pos,
}
constraints = {k: v for k, v in constraint_inputs.items() if v.strip()}
pos_constraint_df = None
if constraints:
    pos_constraint_df = make_ensemble_constraints(
        {pdb_name: constraints},
        {pdb_name: all_conformers},
    )

configure_notebook_warning_filters()
ensemble_results = model.ensemble_sample(
    pdb_to_conformers={pdb_name: all_conformers},
    num_seqs_per_pdb=num_seqs,
    batch_size=1,
    temperature=temperature,
    omit_aas=omit_list,
    num_workers=num_workers,
    out_dir="outputs/seq_des_ensemble",
    pos_constraint_df=pos_constraint_df,
    use_primary_res_type=True,
)

df_ensemble_results = pd.DataFrame(ensemble_results)
title = f"Designed {len(df_ensemble_results)} ensemble-conditioned sequences"
display(HTML(f"<h3>{title}</h3>"))
print(f"Output CIF files saved to outputs/seq_des_ensemble/")
print(f"Available columns: {list(df_ensemble_results.columns)}\n")

df_display = df_ensemble_results[["example_id", "seq", "U"]].copy()
df_display["U"] = df_display["U"].round(2)
df_display["seq"] = df_display["seq"].str[:40] + df_display["seq"].apply(
    lambda s: f"... ({len(s)} aa)" if len(s) > 40 else ""
)
display(df_display.style.set_properties(**{
    "text-align": "left",
    "font-family": "monospace",
}).set_properties(subset=["U"], **{
    "font-weight": "bold",
}).hide(axis="index"))

In [ ]:
#@title Download ensemble-conditioned design outputs
#@markdown Downloads the ensemble-conditioned design outputs from `outputs/seq_des_ensemble/`.

from pathlib import Path
import zipfile
from google.colab import files

outputs_root = Path("outputs")
output_dirs = [outputs_root / "seq_des_ensemble"]
missing_dirs = [str(path) for path in output_dirs if not path.exists()]
if missing_dirs:
    raise FileNotFoundError(f"Run ensemble-conditioned design first to create: {', '.join(missing_dirs)}")

# Save results DataFrame as CSV
df_ensemble_results.to_csv(outputs_root / "seq_des_ensemble" / "seq_des_outputs.csv", index=False)

archive_path = Path("caliby_seq_des_ensemble_results.zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for output_dir in output_dirs:
        for path in output_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(outputs_root))

files.download(str(archive_path))
print("Download started. Check your browser's downloads.")

## Self-consistency evaluation

Refold ensemble-conditioned designed sequences with AlphaFold2.

In [ ]:
#@title Run self-consistency evaluation (ensemble)
#@markdown Refolds ensemble-conditioned designed sequences with AlphaFold2.
num_recycles = 3  #@param {type:"integer"}

from pathlib import Path

import pandas as pd
from IPython.display import display, HTML

ensemble_designed_pdbs = df_ensemble_results["out_pdb"].tolist()

configure_notebook_warning_filters()
ensemble_sc_results = model.self_consistency_eval(
    ensemble_designed_pdbs,
    out_dir="outputs/seq_des_ensemble_sc",
    num_models=5,
    num_recycles=num_recycles,
)

ensemble_sc_df = pd.DataFrame([
    {"sample_id": k, **v} for k, v in ensemble_sc_results.items()
])
df_ensemble_sc = df_ensemble_results.copy()
df_ensemble_sc["sample_id"] = df_ensemble_sc["out_pdb"].apply(lambda path: Path(path).stem)
df_ensemble_sc = df_ensemble_sc.merge(ensemble_sc_df, on="sample_id", how="left", validate="one_to_one")

display(HTML("<h3>Self-consistency evaluation (ensemble)</h3>"))
print(f"AF2 predictions saved to outputs/seq_des_ensemble_sc/\n")

df_display = df_ensemble_sc[["sample_id", "example_id", "U", "sc_ca_rmsd", "avg_ca_plddt", "tmalign_score"]].copy()
df_display[["U", "sc_ca_rmsd", "tmalign_score"]] = df_display[["U", "sc_ca_rmsd", "tmalign_score"]].round(3)
df_display["avg_ca_plddt"] = df_display["avg_ca_plddt"].round(1)
display(df_display.style.set_properties(**{
    "text-align": "left",
    "font-family": "monospace",
}).hide(axis="index"))

# Show 3D viewer for best AF2 prediction (by TM-align score).
import molview as mv

valid_scores = df_ensemble_sc["tmalign_score"].dropna()
if not valid_scores.empty:
    best_idx = valid_scores.idxmax()
    best_sample = df_ensemble_sc.loc[best_idx, "sample_id"]
    best_af2_pdb = Path(f"outputs/seq_des_ensemble_sc/struct_preds/af2_{best_sample}.pdb")

    if best_af2_pdb.exists():
        pdb_data = best_af2_pdb.read_text()
        v = mv.view(width=840, height=500)
        v.addModel(pdb_data, name=f"AF2: {best_sample}")
        v.setColorMode("plddt")
        v.setBackgroundColor("#F2F2F2")
        display(HTML(f"<h4>AF2 prediction: {best_sample} (pLDDT coloring)</h4>"))
        display(HTML(v._repr_html_()))
else:
    print("TM-align scores unavailable; skipping AF2 viewer.")

In [ ]:
#@title Download ensemble self-consistency outputs
#@markdown Downloads only the ensemble-conditioned AlphaFold2 self-consistency outputs.

from pathlib import Path
import zipfile
from google.colab import files

outputs_root = Path("outputs")
sc_dir = outputs_root / "seq_des_ensemble_sc"
aligned_dir = sc_dir / "ca_aligned_struct_preds"
metrics_path = sc_dir / "self_consistency_metrics.csv"
required_paths = [sc_dir, aligned_dir]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Run ensemble self-consistency first to create: {', '.join(missing_paths)}")

# Save self-consistency metrics as CSV
df_ensemble_sc.to_csv(metrics_path, index=False)

archive_path = Path("self_consistency_results.zip")
archive_root = Path("self_consistency_results")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(metrics_path, archive_root / metrics_path.name)
    for path in aligned_dir.rglob("*"):
        if path.is_file():
            zf.write(path, archive_root / path.relative_to(sc_dir))

files.download(str(archive_path))
print("Download started. Check your browser's downloads.")

# Providing your own ensembles for design

You can supply your own set of PDB/CIF conformers instead of generating them with Protpardelle. This is useful when you already have experimental or computationally generated structures for the same protein in different conformational states (e.g., open/closed forms, apo/holo states).

**Option A:** Run the next cell to upload two or more PDB or CIF files as conformers. The first file is the primary conformer, which doesn't affect results but affects which structure designed sequences are threaded onto.

**Option B:** Skip the next cell to use the built-in DHFR example (1rx2 + 1rx4).

In [ ]:
#@title Upload ensemble PDB files (optional)
#@markdown Run this cell to upload two or more PDB or CIF files as conformers. **Skip this cell to use the example DHFR ensemble (1rx2 + 1rx4).**
#@markdown The file that comes first alphabetically will be treated as the primary conformer — rename files to control ordering if needed.
import os, shutil

USER_ENSEMBLE_INPUT_DIR = "user_ensemble_pdbs"
if os.path.exists(USER_ENSEMBLE_INPUT_DIR):
    shutil.rmtree(USER_ENSEMBLE_INPUT_DIR)
os.makedirs(USER_ENSEMBLE_INPUT_DIR)

from google.colab import files
print("Upload two or more PDB or CIF files (conformers):")
uploaded = files.upload()
if len(uploaded) < 2:
    print("Warning: fewer than 2 files uploaded. Upload at least 2 conformers for ensemble-conditioned design.")
for fn in uploaded:
    os.rename(fn, os.path.join(USER_ENSEMBLE_INPUT_DIR, fn))
print(f"Uploaded {len(uploaded)} file(s).")

In [ ]:
#@title Prepare user ensemble
#@markdown Uses uploaded files if present, otherwise falls back to the DHFR example (1rx2 + 1rx4). Cleans and standardizes all conformers.
import os, glob, shutil, base64, urllib.request

from caliby import clean_pdbs
from IPython.display import display, HTML

USER_ENSEMBLE_INPUT_DIR = "user_ensemble_pdbs"
USER_ENSEMBLE_CLEAN_DIR = "user_ensemble_cleaned_pdbs"

os.makedirs(USER_ENSEMBLE_INPUT_DIR, exist_ok=True)
user_pdb_files = sorted(
    glob.glob(os.path.join(USER_ENSEMBLE_INPUT_DIR, "*.pdb"))
    + glob.glob(os.path.join(USER_ENSEMBLE_INPUT_DIR, "*.cif"))
)

if len(user_pdb_files) < 2:
    BASE_URL = "https://raw.githubusercontent.com/ProteinDesignLab/caliby/main/notebooks/data/dhfr"
    _DEFAULT_NAMES = ["1rx2.cif", "1rx4.cif"]
    print("No uploaded ensemble found — downloading DHFR example (1rx2 + 1rx4)...")
    user_pdb_files = []
    for name in _DEFAULT_NAMES:
        dest = os.path.join(USER_ENSEMBLE_INPUT_DIR, name)
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", dest)
        user_pdb_files.append(dest)

if os.path.exists(USER_ENSEMBLE_CLEAN_DIR):
    shutil.rmtree(USER_ENSEMBLE_CLEAN_DIR)

cleaned_paths = clean_pdbs(user_pdb_files, out_dir=USER_ENSEMBLE_CLEAN_DIR)
user_ensemble_conformers = cleaned_paths
user_ensemble_pdb_name = os.path.splitext(os.path.basename(user_ensemble_conformers[0]))[0]

print(f"Primary conformer: {os.path.basename(user_ensemble_conformers[0])}")
print(f"Additional conformers: {[os.path.basename(p) for p in user_ensemble_conformers[1:]]}")
print(f"Total conformers for ensemble design: {len(user_ensemble_conformers)}")

cleaned_b64 = base64.b64encode(open(user_ensemble_conformers[0], "rb").read()).decode()
cleaned_filename = os.path.basename(user_ensemble_conformers[0])
display(HTML(
    "<div style='margin: 10px 0; padding: 12px 16px; background: #fff8e1; border-left: 4px solid #ffb300; border-radius: 4px;'>"
    "<b>Note:</b> All conformers have been cleaned and standardized to mmCIF format. "
    "The primary conformer is listed first. "
    "<b>If you plan to use positional constraints, verify residue numbering in the cleaned primary conformer.</b><br><br>"
    f"<a href='data:chemical/x-cif;base64,{cleaned_b64}' download='{cleaned_filename}' "
    "style='display: inline-block; padding: 6px 14px; background: #ffb300; color: #fff; "
    "border-radius: 4px; text-decoration: none; font-weight: 600;'>Download cleaned primary conformer</a>"
    "</div>"
))

In [ ]:
#@title Run ensemble-conditioned design (user ensemble)
#@markdown ### Sampling parameters
num_seqs = 4  #@param {type:"integer"}
temperature = 0.01  #@param {type:"number"}
omit_aas = ""  #@param {"type":"string","placeholder":"e.g. C,M"}
#@markdown ### Constraints (optional)
#@markdown Enable and fill in the fields you need. Positions use `label_seq_id` numbering relative to the primary conformer.
#@markdown <details><summary><b>Constraint reference</b></summary>
#@markdown
#@markdown | Constraint | Format | Description |
#@markdown |---|---|---|
#@markdown | `fixed_pos_seq` | `A1-10,A20-30` | Fix sequence positions |
#@markdown | `fixed_pos_scn` | `A1-10` | Also condition on sidechains (subset of fixed_pos_seq) |
#@markdown | `fixed_pos_override_seq` | `A26:A,A27:L` | Override then fix position to specific AA |
#@markdown | `pos_restrict_aatype` | `A26:AVG,A27:VG` | Restrict allowed AAs at positions |
#@markdown | `symmetry_pos` | `A10,B10\|A11,B11` | Tie sampling across positions |
#@markdown
#@markdown </details>
fixed_pos_seq = ""  #@param {"type":"string","placeholder":"e.g. A1-100,B1-100"}
fixed_pos_scn = ""  #@param {"type":"string","placeholder":"e.g. A1-10,A12,A15-20"}
fixed_pos_override_seq = ""  #@param {"type":"string","placeholder":"e.g. A26:A,A27:L"}
pos_restrict_aatype = ""  #@param {"type":"string","placeholder":"e.g. A26:AVG,A27:VG"}
symmetry_pos = ""  #@param {"type":"string","placeholder":"e.g. A10,B10,C10|A11,B11,C11"}

from pathlib import Path

import pandas as pd
from IPython.display import display, HTML
from caliby import make_ensemble_constraints

omit_list = [aa.strip() for aa in omit_aas.split(",") if aa.strip()] or None

constraint_inputs = {
    "fixed_pos_seq": fixed_pos_seq,
    "fixed_pos_scn": fixed_pos_scn,
    "fixed_pos_override_seq": fixed_pos_override_seq,
    "pos_restrict_aatype": pos_restrict_aatype,
    "symmetry_pos": symmetry_pos,
}
constraints = {k: v for k, v in constraint_inputs.items() if v.strip()}
pos_constraint_df = None
if constraints:
    pos_constraint_df = make_ensemble_constraints(
        {user_ensemble_pdb_name: constraints},
        {user_ensemble_pdb_name: user_ensemble_conformers},
    )

configure_notebook_warning_filters()
user_ensemble_results = model.ensemble_sample(
    pdb_to_conformers={user_ensemble_pdb_name: user_ensemble_conformers},
    num_seqs_per_pdb=num_seqs,
    batch_size=1,
    temperature=temperature,
    omit_aas=omit_list,
    num_workers=num_workers,
    out_dir="outputs/seq_des_user_ensemble",
    pos_constraint_df=pos_constraint_df,
    use_primary_res_type=True,
)

df_user_ensemble_results = pd.DataFrame(user_ensemble_results)
title = f"Designed {len(df_user_ensemble_results)} ensemble-conditioned sequences (user ensemble)"
display(HTML(f"<h3>{title}</h3>"))
print(f"Output CIF files saved to outputs/seq_des_user_ensemble/")
print(f"Available columns: {list(df_user_ensemble_results.columns)}\n")

df_display = df_user_ensemble_results[["example_id", "seq", "U"]].copy()
df_display["U"] = df_display["U"].round(2)
df_display["seq"] = df_display["seq"].str[:40] + df_display["seq"].apply(
    lambda s: f"... ({len(s)} aa)" if len(s) > 40 else ""
)
display(df_display.style.set_properties(**{
    "text-align": "left",
    "font-family": "monospace",
}).set_properties(subset=["U"], **{
    "font-weight": "bold",
}).hide(axis="index"))

In [ ]:
#@title Download user ensemble design outputs
#@markdown Downloads the user-ensemble-conditioned design outputs from `outputs/seq_des_user_ensemble/`.

from pathlib import Path
import zipfile
from google.colab import files

outputs_root = Path("outputs")
output_dirs = [outputs_root / "seq_des_user_ensemble"]
missing_dirs = [str(path) for path in output_dirs if not path.exists()]
if missing_dirs:
    raise FileNotFoundError(f"Run user ensemble design first to create: {', '.join(missing_dirs)}")

# Save results DataFrame as CSV
df_user_ensemble_results.to_csv(outputs_root / "seq_des_user_ensemble" / "seq_des_outputs.csv", index=False)

archive_path = Path("caliby_seq_des_user_ensemble_results.zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for output_dir in output_dirs:
        for path in output_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(outputs_root))

files.download(str(archive_path))
print("Download started. Check your browser's downloads.")

## Self-consistency evaluation

Refold user-ensemble-conditioned designed sequences with AlphaFold2.

In [ ]:
#@title Run self-consistency evaluation (user ensemble)
#@markdown Refolds user-ensemble-conditioned designed sequences with AlphaFold2.
num_recycles = 3  #@param {type:"integer"}

from pathlib import Path

import pandas as pd
from IPython.display import display, HTML

user_ensemble_designed_pdbs = df_user_ensemble_results["out_pdb"].tolist()

configure_notebook_warning_filters()
user_ensemble_sc_results = model.self_consistency_eval(
    user_ensemble_designed_pdbs,
    out_dir="outputs/seq_des_user_ensemble_sc",
    num_models=5,
    num_recycles=num_recycles,
)

user_ensemble_sc_df = pd.DataFrame([
    {"sample_id": k, **v} for k, v in user_ensemble_sc_results.items()
])
df_user_ensemble_sc = df_user_ensemble_results.copy()
df_user_ensemble_sc["sample_id"] = df_user_ensemble_sc["out_pdb"].apply(lambda path: Path(path).stem)
df_user_ensemble_sc = df_user_ensemble_sc.merge(user_ensemble_sc_df, on="sample_id", how="left", validate="one_to_one")

display(HTML("<h3>Self-consistency evaluation (user ensemble)</h3>"))
print(f"AF2 predictions saved to outputs/seq_des_user_ensemble_sc/\n")

df_display = df_user_ensemble_sc[["sample_id", "example_id", "U", "sc_ca_rmsd", "avg_ca_plddt", "tmalign_score"]].copy()
df_display[["U", "sc_ca_rmsd", "tmalign_score"]] = df_display[["U", "sc_ca_rmsd", "tmalign_score"]].round(3)
df_display["avg_ca_plddt"] = df_display["avg_ca_plddt"].round(1)
display(df_display.style.set_properties(**{
    "text-align": "left",
    "font-family": "monospace",
}).hide(axis="index"))

# Show 3D viewer for best AF2 prediction (by TM-align score).
import molview as mv

valid_scores = df_user_ensemble_sc["tmalign_score"].dropna()
if not valid_scores.empty:
    best_idx = valid_scores.idxmax()
    best_sample = df_user_ensemble_sc.loc[best_idx, "sample_id"]
    best_af2_pdb = Path(f"outputs/seq_des_user_ensemble_sc/struct_preds/af2_{best_sample}.pdb")

    if best_af2_pdb.exists():
        pdb_data = best_af2_pdb.read_text()
        v = mv.view(width=840, height=500)
        v.addModel(pdb_data, name=f"AF2: {best_sample}")
        v.setColorMode("plddt")
        v.setBackgroundColor("#F2F2F2")
        display(HTML(f"<h4>AF2 prediction: {best_sample} (pLDDT coloring)</h4>"))
        display(HTML(v._repr_html_()))
else:
    print("TM-align scores unavailable; skipping AF2 viewer.")

In [ ]:
#@title Download user ensemble self-consistency outputs
#@markdown Downloads only the user-ensemble AlphaFold2 self-consistency outputs.

from pathlib import Path
import zipfile
from google.colab import files

outputs_root = Path("outputs")
sc_dir = outputs_root / "seq_des_user_ensemble_sc"
aligned_dir = sc_dir / "ca_aligned_struct_preds"
metrics_path = sc_dir / "self_consistency_metrics.csv"
required_paths = [sc_dir, aligned_dir]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Run user ensemble self-consistency first to create: {', '.join(missing_paths)}")

# Save self-consistency metrics as CSV
df_user_ensemble_sc.to_csv(metrics_path, index=False)

archive_path = Path("user_ensemble_self_consistency_results.zip")
archive_root = Path("self_consistency_results")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(metrics_path, archive_root / metrics_path.name)
    for path in aligned_dir.rglob("*"):
        if path.is_file():
            zf.write(path, archive_root / path.relative_to(sc_dir))

files.download(str(archive_path))
print("Download started. Check your browser's downloads.")